In [1]:
import pandas as pd
import numpy as np
from datetime import datetime, timedelta

In [2]:
import kagglehub

path = kagglehub.dataset_download("lakshmi25npathi/online-retail-dataset")

print("Path to dataset files:", path)

100%|██████████| 43.3M/43.3M [00:02<00:00, 21.0MB/s]

Extracting files...


Path to dataset files: /root/.cache/kagglehub/datasets/lakshmi25npathi/online-retail-dataset/versions/1


In [3]:
import os


file_list = os.listdir(path)
print(f"Files in the dataset directory: {file_list}")

data_file = os.path.join(path, 'online_retail_II.xlsx')

df = pd.read_excel(data_file)

display(df.head())

Files in the dataset directory: ['online_retail_II.xlsx']


,Invoice,StockCode,Description,Quantity,InvoiceDate,Price,Customer ID,Country
0,489434,85048,15CM CHRISTMAS GLASS BALL 20 LIGHTS,12,2009-12-01 07:45:00,6.95,13085.0,United Kingdom
1,489434,79323P,PINK CHERRY LIGHTS,12,2009-12-01 07:45:00,6.75,13085.0,United Kingdom
2,489434,79323W,WHITE CHERRY LIGHTS,12,2009-12-01 07:45:00,6.75,13085.0,United Kingdom
3,489434,22041,"RECORD FRAME 7"" SINGLE SIZE",48,2009-12-01 07:45:00,2.10,13085.0,United Kingdom
4,489434,21232,STRAWBERRY CERAMIC TRINKET BOX,24,2009-12-01 07:45:00,1.25,13085.0,United Kingdom


In [4]:
df['InvoiceDate'] = pd.to_datetime(df['InvoiceDate'], errors='coerce')


min_date = df['InvoiceDate'].min()
print(f"\nOriginal Date Range: {min_date} to {df['InvoiceDate'].max()}")


date_shift = timedelta(days=365 * 14)
df['InvoiceDate'] = df['InvoiceDate'] + date_shift

print(f"Modernized Date Range: {df['InvoiceDate'].min()} to {df['InvoiceDate'].max()}")


Original Date Range: 2009-12-01 07:45:00 to 2010-12-09 20:01:00
Modernized Date Range: 2023-11-28 07:45:00 to 2024-12-05 20:01:00


In [5]:
df.isna().value_counts()

Invoice  StockCode  Description  Quantity  InvoiceDate  Price  Customer ID  Country
False    False      False        False     False        False  False        False      417534
                                                               True         False      104999
                    True         False     False        False  True         False        2928
Name: count, dtype: int64

In [6]:
df.isnull().sum()

,0
Invoice,0
StockCode,0
Description,2928
Quantity,0
InvoiceDate,0
Price,0
Customer ID,107927
Country,0


In [7]:
#Data Cleaning

# Check missing Customer IDs
missing_customer_ids = df['Customer ID'].isnull().sum()
print(f"Found {missing_customer_ids} rows with NULL Customer ID")


if missing_customer_ids > 0:
    print("Generating synthetic Customer IDs for anonymous transactions...")

    # Get the maximum existing Customer ID to avoid collisions
    max_real_id = df['Customer ID'].max()
    synthetic_start_id = max(900000, int(max_real_id) + 1000)

    # Generate unique synthetic IDs
    synthetic_ids = range(synthetic_start_id, synthetic_start_id + missing_customer_ids)

    # Assign synthetic IDs to null Customer IDs
    df.loc[df['Customer ID'].isnull(), 'Customer ID'] = list(synthetic_ids)

    print(f"✓ Generated {missing_customer_ids} synthetic Customer IDs (range: {synthetic_start_id} to {synthetic_start_id + missing_customer_ids - 1})")


df['Customer ID'] = df['Customer ID'].astype(int)


initial_rows = len(df)
df = df[df['Customer ID'].notna()]
if initial_rows > len(df):
    print(f"Removed {initial_rows - len(df)} rows with NULL Customer ID")


initial_rows = len(df)
df = df[df['InvoiceDate'].notna()]
if initial_rows > len(df):
    print(f"Removed {initial_rows - len(df)} rows with NULL InvoiceDate")

# Drop unnecessary columns (Description not needed for RFM)
df = df[['Invoice', 'Quantity', 'InvoiceDate', 'Price', 'Customer ID', 'Country']]

df['Amount'] = df['Quantity'] * df['Price']

# Remove negative or zero amounts (returns/cancellations)
initial_rows = len(df)
df = df[df['Amount'] > 0]
print(f"Removed {initial_rows - len(df)} rows with negative/zero amounts")

# Remove negative quantities
df = df[df['Quantity'] > 0]

print(f"\nCleaned Data Shape: {df.shape}")

Found 107927 rows with NULL Customer ID
Generating synthetic Customer IDs for anonymous transactions...
✓ Generated 107927 synthetic Customer IDs (range: 900000 to 1007926)
Removed 13895 rows with negative/zero amounts

Cleaned Data Shape: (511566, 7)


In [8]:
df.isna().sum()

,0
Invoice,0
Quantity,0
InvoiceDate,0
Price,0
Customer ID,0
Country,0
Amount,0


In [10]:
#rfm calculations


# Using the maximum date in the dataset + 1 day
analysis_date = df['InvoiceDate'].max() + timedelta(days=1)
print(f"Analysis Date (Reference Point): {analysis_date}")

# Group by CustomerID and calculate RFM metrics
rfm = df.groupby('Customer ID').agg({
    'InvoiceDate': lambda x: (analysis_date - x.max()).days,  # Recency
    'Invoice': 'nunique',  # Frequency (unique invoices)
    'Amount': 'sum'  # Monetary
}).reset_index()

# Rename columns
rfm.columns = ['CustomerID', 'Recency', 'Frequency', 'Monetary']

print("\nRFM Metrics Calculated:")
print(rfm.head(10))
print("\nRFM Summary Statistics:")
print(rfm.describe())
print (rfm.describe )


Analysis Date (Reference Point): 2024-12-06 20:01:00

RFM Metrics Calculated:
   CustomerID  Recency  Frequency  Monetary
0       12346      165         11    372.86
1       12347        3          2   1323.32
2       12348       74          1    222.16
3       12349       43          3   2671.14
4       12351       11          1    300.93
5       12352       11          2    343.80
6       12353       44          1    317.76
7       12355      203          1    488.21
8       12356       16          3   3562.25
9       12357       24          2  12079.99

RFM Summary Statistics:
         CustomerID        Recency      Frequency       Monetary
count  1.082140e+05  108214.000000  108214.000000  108214.000000
mean   9.166975e+05     171.124448       1.137699      95.239669
std    1.861677e+05     125.626241       1.765270    1827.774413
min    1.234600e+04       1.000000       1.000000       0.140000
25%    9.232672e+05      45.000000       1.000000       2.980000
50%    9.521365e+05    

In [11]:
#rfm scoring



rfm['R_Score'] = pd.qcut(rfm['Recency'], q=4, labels=[4, 3, 2, 1])
rfm['F_Score'] = pd.qcut(rfm['Frequency'].rank(method='first'), q=4, labels=[1, 2, 3, 4])
rfm['M_Score'] = pd.qcut(rfm['Monetary'].rank(method='first'), q=4, labels=[1, 2, 3, 4])

# Convert scores to integers
rfm['R_Score'] = rfm['R_Score'].astype(int)
rfm['F_Score'] = rfm['F_Score'].astype(int)
rfm['M_Score'] = rfm['M_Score'].astype(int)

# Create combined RFM score
rfm['RFM_Score'] = rfm['R_Score'].astype(str) + rfm['F_Score'].astype(str) + rfm['M_Score'].astype(str)

print("\nRFM Scores Added:")
print(rfm.head(10))


RFM Scores Added:
   CustomerID  Recency  Frequency  Monetary  R_Score  F_Score  M_Score  \
0       12346      165         11    372.86        3        4        4   
1       12347        3          2   1323.32        4        4        4   
2       12348       74          1    222.16        3        1        4   
3       12349       43          3   2671.14        4        4        4   
4       12351       11          1    300.93        4        1        4   
5       12352       11          2    343.80        4        4        4   
6       12353       44          1    317.76        4        1        4   
7       12355      203          1    488.21        2        1        4   
8       12356       16          3   3562.25        4        4        4   
9       12357       24          2  12079.99        4        4        4   

  RFM_Score  
0       344  
1       444  
2       314  
3       444  
4       414  
5       444  
6       414  
7       214  
8       444  
9       444  


In [12]:
# customer segmentation

def segment_customers(row):
    """Segment customers based on RFM scores"""
    r, f, m = row['R_Score'], row['F_Score'], row['M_Score']
    score = row['RFM_Score']

    # Champions: Best customers
    if r >= 4 and f >= 4 and m >= 4:
        return 'Champions'

    # Loyal Customers: High frequency, moderate recency
    elif r >= 3 and f >= 4:
        return 'Loyal Customers'

    # Potential Loyalists: Recent customers with potential
    elif r >= 4 and f >= 2 and f <= 3:
        return 'Potential Loyalists'

    # New Customers: Recent but low frequency
    elif r >= 4 and f == 1:
        return 'New Customers'

    # At Risk: Were good customers but haven't purchased recently
    elif r <= 2 and f >= 3 and m >= 3:
        return 'At Risk'

    # Can't Lose Them: High value but long time since purchase
    elif r <= 2 and f >= 4 and m >= 4:
        return "Can't Lose Them"

    # Hibernating: Low recency, low frequency, but some history
    elif r <= 2 and f <= 2:
        return 'Hibernating'

    # Lost: Lowest scores across the board
    elif r == 1 and f <= 2:
        return 'Lost'

    # About to Sleep: Below average recency and frequency
    elif r <= 3 and f <= 3:
        return 'About to Sleep'

    # Promising: Recent customers with moderate engagement
    else:
        return 'Promising'

rfm['Segment'] = rfm.apply(segment_customers, axis=1)

segment_distribution = rfm['Segment'].value_counts().sort_values(ascending=False)
print("\nCustomer Segment Distribution:")
print(segment_distribution)
print(f"\nPercentage Distribution:")
print((segment_distribution / len(rfm) * 100).round(2))



Customer Segment Distribution:
Segment
Hibernating            53305
About to Sleep         26172
Loyal Customers        19213
Champions               7551
Potential Loyalists     1227
At Risk                  411
New Customers            335
Name: count, dtype: int64

Percentage Distribution:
Segment
Hibernating            49.26
About to Sleep         24.19
Loyal Customers        17.75
Champions               6.98
Potential Loyalists     1.13
At Risk                 0.38
New Customers           0.31
Name: count, dtype: float64


In [13]:
# exploring data

# Select final columns for export
rfm_export = rfm[['CustomerID', 'Recency', 'Frequency', 'Monetary',
                   'R_Score', 'F_Score', 'M_Score', 'RFM_Score', 'Segment']]

rfm_export['Monetary'] = rfm_export['Monetary'].round(2)


output_file = 'rfm_analysis_clean.csv'
rfm_export.to_csv(output_file, index=False)
print(f"\n Data exported to: {output_file}")
print(f" Total Customers Analyzed: {len(rfm_export)}")

print("\nSample of Final Output:")
print(rfm_export.head(15))


 Data exported to: rfm_analysis_clean.csv
 Total Customers Analyzed: 108214

Sample of Final Output:
    CustomerID  Recency  Frequency  Monetary  R_Score  F_Score  M_Score  \
0        12346      165         11    372.86        3        4        4   
1        12347        3          2   1323.32        4        4        4   
2        12348       74          1    222.16        3        1        4   
3        12349       43          3   2671.14        4        4        4   
4        12351       11          1    300.93        4        1        4   
5        12352       11          2    343.80        4        4        4   
6        12353       44          1    317.76        4        1        4   
7        12355      203          1    488.21        2        1        4   
8        12356       16          3   3562.25        4        4        4   
9        12357       24          2  12079.99        4        4        4   
10       12358       11          3   2719.01        4        4        4  

In [15]:
print("\nSEGMENT INSIGHTS ")
segment_insights = rfm.groupby('Segment').agg({
    'Recency': 'mean',
    'Frequency': 'mean',
    'Monetary': ['mean', 'sum'],
    'CustomerID': 'count'
}).round(2)

segment_insights.columns = ['Avg_Recency', 'Avg_Frequency', 'Avg_Monetary', 'Total_Revenue', 'Customer_Count']
segment_insights = segment_insights.sort_values('Total_Revenue', ascending=False)

print(segment_insights)

print("\n Ready for SQL import")


SEGMENT INSIGHTS 
                     Avg_Recency  Avg_Frequency  Avg_Monetary  Total_Revenue  \
Segment                                                                        
Champions                  17.57           2.51        902.11     6811816.45   
Loyal Customers            21.70           1.15         83.61     1606307.52   
Hibernating               284.27           1.00         16.37      872746.19   
About to Sleep            101.86           1.00         20.60      539031.70   
At Risk                   213.78           2.36        856.99      352223.96   
New Customers              24.58           1.00        339.87      113856.21   
Potential Loyalists        43.88           1.00          8.38       10283.56   

                     Customer_Count  
Segment                              
Champions                      7551  
Loyal Customers               19213  
Hibernating                   53305  
About to Sleep                26172  
At Risk                         